In [ ]:
import os
import time
import pandas as pd
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate

os.environ["GOOGLE_API_KEY"] = "api key here"  # Replace with your actual API key

# 1. Setup Environment
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.1)

# 2. Load Vector Database
print("Loading FAISS Database...")
vector_store = FAISS.load_local("../data/faiss_index", embeddings, allow_dangerous_deserialization=True)
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# 3. Test Suite
test_queries = [
    # --- Exact Fact & Indicator Retrieval ---
    "What specific CVE does APT29 use to attack Embassies?",
    "Which ransomware group explicitly exploited the Citrix Bleed vulnerability (CVE-2023-4966)?",
    "What are the specific initial access vectors utilized by the Scattered Spider group?",
    "What are the historical Indicators of Compromise (IOCs) associated with Stuxnet 0.5?",
    
    # --- Behavioral & Conceptual Synthesis ---
    "Describe the living-off-the-land techniques used by Volt Typhoon to maintain persistence.",
    "How does the Akira ransomware group handle data exfiltration and double extortion?",
    "What specific remote monitoring and management (RMM) tools are associated with MuddyWater campaigns?",
    "Explain the role of legitimate IT tools in recent Iranian state-sponsored cyber operations.",
    
    # --- Guardrail Stress Tests (Should trigger 'Insufficient data') ---
    "What lateral movement techniques does the LockBit 3.0 group use against macOS environments?", # Plausible but likely missing
    "Describe the primary attack vectors used by the fictional APT-99 'Neon Mirage' group targeting lunar outposts." # Impossible
]

results = []

# 4. Run Evaluation
print("Running Inference Evaluation...\n")
for query in test_queries:
    start_time = time.time()
    
    # Retrieve context
    docs = retriever.invoke(query)
    context = "\n".join([doc.page_content for doc in docs])
    
    # Simple prompt for testing
    prompt = f"Context: {context}\n\nQuestion: {query}\nAnswer based ONLY on context. If not found, say 'Insufficient data'."
    
    # Generate
    response = llm.invoke(prompt).content
    latency = round(time.time() - start_time, 2)
    
    # Record metrics
    results.append({
        "Query": query,
        "Retrieval Chunk Count": len(docs),
        "Inference Latency (sec)": latency,
        "Guardrail Triggered": "Insufficient data" in response,
        "Response Snippet": response[:100] + "..."
    })

# 5. Export Results
df = pd.DataFrame(results)
df.to_csv("../results/early_evaluation_metrics.csv", index=False)
print("Evaluation complete. Metrics saved to results/early_evaluation_metrics.csv")
display(df)

Loading FAISS Database...
Running Inference Evaluation...

Evaluation complete. Metrics saved to results/early_evaluation_metrics.csv


,Query,Retrieval Chunk Count,Inference Latency (sec),Guardrail Triggered,Response Snippet
0,What specific CVE does APT29 use to attack Emb...,3,2.77,True,Insufficient data....
1,Which ransomware group explicitly exploited th...,3,2.29,False,LockBit 3.0 ransomware (and its affiliates) ex...
2,What are the specific initial access vectors u...,3,3.28,False,The specific initial access vectors utilized b...
3,What are the historical Indicators of Compromi...,3,2.46,True,Insufficient data....
4,Describe the living-off-the-land techniques us...,3,4.20,True,Insufficient data. The provided text states th...
5,How does the Akira ransomware group handle dat...,3,23.35,False,The Akira ransomware group handles data exfilt...
6,What specific remote monitoring and management...,3,20.99,False,ConnectWise and RemoteUtilities....
7,Explain the role of legitimate IT tools in rec...,3,6.17,True,Insufficient data....
8,What lateral movement techniques does the Lock...,3,4.48,True,Insufficient data....
9,Describe the primary attack vectors used by th...,3,2.87,True,Insufficient data....
